# Regime-Shift: Macro-Aware Tactical Asset Allocation Engine

Final notebook for the Summer of Quant Advanced Project.

This notebook runs the full pipeline top-to-bottom:

```text
data → features → HMM regime detection → convex optimization → walk-forward backtest → results
```

Run all cells after installing `requirements.txt`.

## 1. Imports and configuration

The heavy implementation is kept in `regime_shift_pipeline.py` so the same code can be run both as a notebook and as a reproducible Python script.

In [ ]:
from regime_shift_pipeline import Config, run_pipeline

# Submission run: use real yfinance data.
# Keep allow_synthetic_fallback=False for final reported results.
config = Config(
    start_date="2015-01-01",
    end_date=None,
    initial_train_days=756,
    test_days=252,
    transaction_cost_bps=7.5,
    allow_synthetic_fallback=False,
)
config

## 2. Run the full pipeline

This cell downloads data, builds features, trains the HMM inside walk-forward folds, optimizes regime-specific weights with CVXPY, backtests the strategy, and saves all required submission outputs under `outputs/`.

In [ ]:
results = run_pipeline(config)

## 3. Final HMM transition matrix

This is the transition probability matrix from the final HMM fit. The walk-forward backtest does **not** use this full-sample model for performance; it uses fold-by-fold HMM re-fitting.

In [ ]:
results["transition_matrix"].round(3)

## 4. Regime-specific optimized weights

In [ ]:
results["weight_table"].round(4)

## 5. Performance summary

The net strategy includes transaction costs. Benchmarks have zero turnover in this implementation.

In [ ]:
summary = results["summary"].copy()
summary.round(4)

## 6. Saved plots

After the pipeline runs, open these generated files:

- `outputs/plots/final_hmm_regime_overlay.png`
- `outputs/plots/walk_forward_regime_overlay.png`
- `outputs/plots/equity_curves.png`
- `outputs/plots/drawdowns.png`
- `outputs/plots/dynamic_weights.png`

These cover the required regime chart, equity curve chart, drawdown chart, and portfolio weight evolution.

In [ ]:
from pathlib import Path
from IPython.display import Image, display

plot_paths = [
    Path("outputs/plots/final_hmm_regime_overlay.png"),
    Path("outputs/plots/walk_forward_regime_overlay.png"),
    Path("outputs/plots/equity_curves.png"),
    Path("outputs/plots/drawdowns.png"),
    Path("outputs/plots/dynamic_weights.png"),
]

for path in plot_paths:
    if path.exists():
        print(path)
        display(Image(filename=str(path)))
    else:
        print(f"Not found yet: {path}. Run the pipeline cell first.")

## 7. Optional offline smoke test

Only use this when debugging on a machine without internet. Synthetic data is **not** for final reported project results.

```bash
FORCE_SYNTHETIC_DATA=1 ALLOW_SYNTHETIC_FALLBACK=1 python regime_shift_pipeline.py
```